This notebook serves as a starting point for the final project of the Introduction to Machine Learning 2024/25 course. You must use exactly the code provided to load and split the dataset.

# Code to Load the Dataset

In [ ]:
!pip install -U "datasets==3.6.0"
!pip install gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 8.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 

In [ ]:
# Import of the libraries

from transformers import  GPT2TokenizerFast, GPT2Config, GPT2LMHeadModel
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer
from transformers import PreTrainedTokenizerFast
import random
import torch
from torch.utils.data import DataLoader
import torch.nn.functional as F
import math

# Import of the link to the drive

from google.colab import drive

import gdown

In [ ]:
### YOU MUST NOT CHANGE THIS CELL! ###

from datasets import load_dataset

full_dataset = load_dataset("skeskinen/TinyStories-GPT4", split="train")
full_dataset = full_dataset.remove_columns([c for c in full_dataset.column_names if c not in ["story", "features"]])
assert len(full_dataset) == 2745100

splits = full_dataset.train_test_split(test_size=10000, seed=42, shuffle=True)

train_dataset = splits["train"]
test_dataset  = splits["test"]

assert len(train_dataset) == 2735100
assert len(test_dataset)  == 10000

assert train_dataset[0]["story"][:33] == "One day, a little girl named Lily"
assert train_dataset[0]["features"] == ["Dialogue", "Conflict"]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/554 [00:00<?, ?B/s]

(…)-00000-of-00008-c63ccd5d5290f4a1.parquet:   0%|          | 0.00/194M [00:00<?, ?B/s]

(…)-00001-of-00008-478199d8ac044910.parquet:   0%|          | 0.00/194M [00:00<?, ?B/s]

(…)-00002-of-00008-9b868f59be94d815.parquet:   0%|          | 0.00/194M [00:00<?, ?B/s]

...

(…)-00004-of-00008-5f8ac0bb66de5834.parquet:   0%|          | 0.00/194M [00:00<?, ?B/s]

(…)-00005-of-00008-e8c22c3e776b87dd.parquet:   0%|          | 0.00/194M [00:00<?, ?B/s]

(…)-00006-of-00008-941f57106aca3340.parquet:   0%|          | 0.00/194M [00:00<?, ?B/s]

(…)-00007-of-00008-771d8aa2d5ce5c95.parquet:   0%|          | 0.00/194M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2745100 [00:00<?, ? examples/s]

In [ ]:
# Print of the first example of our training set

from pprint import pprint

pprint(train_dataset[0])

{'features': ['Dialogue', 'Conflict'],
 'story': 'One day, a little girl named Lily went for a walk with her mom. '
          'They saw a big cliff near the water. Lily was fearful and did not '
          "want to go near the cliff. She held her mom's hand tight.\n"
          '"Mom, I am scared," Lily said. Her mom looked at her and smiled. '
          '"It\'s okay to be scared, but I will tell you a secret. When I am '
          'scared, I take a deep breath and count to three. Then, I feel '
          'better."\n'
          'Lily tried what her mom said. She took a deep breath and counted to '
          'three. She felt better and was not as scared. Then, a big bird flew '
          'by and almost hit Lily. She was scared again, but her mom was there '
          'to help her.\n'
          '"Remember what I told you, Lily. Take a deep breath and count to '
          'three," her mom said. Lily did it again, and she felt better. They '
          'walked away from the cliff, and Lily wa

In [ ]:
# Choose a sub-sample size for our training set, remembering that the maximum number of stories are 2735100

SUBSAMPLED_TRAINSET_SIZE = 160000                                                # DON'T CHANGE THIS HYPERPARAMETER

# Create a validation set out of the training set
splits = train_dataset.train_test_split(test_size=10000, seed=42, shuffle=True)

train_dataset = splits["train"]
val_dataset   = splits["test"]

# First test on a subset of the training set
train_dataset = train_dataset.select(range(SUBSAMPLED_TRAINSET_SIZE))

# Choice of model and dictionary

In [ ]:
# Configuration fot our model

custom_tokenizer = True                            # Tokenizer using the Byte-Pair Encoding tokenization (BPE)

do_train = False

# Set of the Hyperparameters

epochs     = 1                                     # Number of epochs chosen for the training phase
batch_size = 16                                    # Dimension of the batch
max_length = 512                                   # Maximum length

In [ ]:
# BPE tokenization

if custom_tokenizer == True:


    vocab_size = 8192                                                                                                                                      # Size of the vocabulary for the tokenization

    # Create a new BPE tokenizer
    tokenizer_obj               = Tokenizer(BPE(unk_token="<|unk|>"))
    tokenizer_obj.pre_tokenizer = Whitespace()

    trainer                     = BpeTrainer(vocab_size=vocab_size, special_tokens=["<|eos|>", "<|pad|>", "<|unk|>"])                                      # Define a trainer with the desired vocabulary size and special tokens

    # Train the tokenizer on the TinyStories dataset (using the 'story' field from train_dataset)
    # Note: if memory is a concern, consider sampling a subset of train_dataset
    stories_iter                = (example["story"] for example in train_dataset)

    tokenizer_obj.train_from_iterator(stories_iter, trainer=trainer)

    # Convert the trained tokenizer to a Hugging Face PreTrainedTokenizerFast
    tokenizer                   = PreTrainedTokenizerFast(tokenizer_object=tokenizer_obj, eos_token="<|eos|>", unk_token="<|unk|>", pad_token="<|pad|>")    # Convert the trained tokenizer to a Hugging Face PreTrainedTokenizerFast
else:

    # gpt-2 tokenizer
    tokenizer                   = GPT2TokenizerFast.from_pretrained('gpt2', add_bos_token=True)
    vocab_size                  = len(tokenizer)

In [ ]:
# Adding of the features tag-tokens
tokenizer.add_tokens(["<|BadEnding|>", "<|Conflict|>","<|Dialogue|>", "<|Foreshadowing|>","<|MoralValue|>", "<|Twist|>"])  #adding the features tag

6

In [ ]:
# New length of the vocabulary after addition of the features tokens
len(tokenizer)

8198

In [ ]:
 # Define our model using the structure of gpt-2
config = GPT2Config(
    #
    vocab_size    = len(tokenizer),            # Size of the vocabulary with the addition of the feature tokens

    n_positions   = 2048,                      # Max number of positions in each input sequence (context length).

    n_embd        = 256,                       # Size of the hidden embedding vectors.

    n_layer       = 2,                         # Number of Transformer decoder blocks (i.e., layers).

    n_head        = 4,                         # Number of self-attention heads in each layer.

    eos_token_id  = tokenizer.eos_token_id,    # ID used to represent the end-of-sequence token.

     pad_token_id = tokenizer.pad_token_id     # ID used to represent padding token.

   )


model = GPT2LMHeadModel(config)

if hasattr(model.config, "loss_type"):         # Function that returns true if findloss_type = 'None'.
    delattr(model.config, "loss_type")         # delattr() function will delete the specified attribute from the specified object.

# Print the number of parameters in the model
print("Number of parameters in the model:", model.num_parameters())

Number of parameters in the model: 4203008


In [ ]:
# Add to our dictionary the pad token (if it doesn't exist) and "add" it to the model
if tokenizer.pad_token is None:
        #
        print(f"Setting pad_token to <|pad|>, its id is the eos_token_id")
        tokenizer.add_special_tokens({'pad_token': '<|pad|>'})
        model.resize_token_embeddings(len(tokenizer))

In [ ]:
# Printing the token IDs, assigned by the BPE tokenizer,of  two special tokens:
print("Tokenizer EOS id:", tokenizer.eos_token_id)
print("Tokenizer PAD id:", tokenizer.pad_token_id)

Tokenizer EOS id: 0
Tokenizer PAD id: 1


# Pre-Training Set Up

In [ ]:
def preprocess(example):
    #
    tok_feat = ""

    for feat in example["features"]:
        #
        tok_feat = tok_feat + "<|"+feat+"|>" + " "

    text = tok_feat + example["story"] + tokenizer.eos_token

    # Tokenize with truncation (padding deferred to collate_fn)
    return tokenizer(text, truncation=True, max_length=max_length)

# Preprocess the training dataset so that each example has the input_ids with prepended feature tokens
train_dataset = train_dataset.map(preprocess)
val_dataset   = val_dataset.map(preprocess)


Map:   0%|          | 0/160000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
def pad_lists(DataSet, max_length):

  # This function takes a list of tokenized sequences and makes sure every sequence is of the max_length
  # of tokens (fixed above) padding the shorter sequences with the soecial token pad <|pad|>

  #  Input:
  #      DataSet    : List of tokenized sequences
  #      max_length : Desired length of each sequence after the padding

  #  Output:
  #     Tokenized sequences each of dimension equal to max_length, some are padded


    padded_sent = []                                                                  # Empty vector that collects the padded sequences

    for sentence in DataSet:
        #
        padding_length = max_length - len(sentence)                                   # Setting of the padding length


        if padding_length > 0:
            #
            padded_inner_list = sentence + [tokenizer.pad_token_id] * padding_length  # Adding of the padding sequence
        else:
            #
            padded_inner_list = sentence                                              # No-padding if the sequence is already of the max length

        padded_sent.append(padded_inner_list)                                         # Adding of the padded sequence to the vector

    return padded_sent


ignore_tokens = [tokenizer.pad_token_id] + tokenizer.convert_tokens_to_ids(["<|BadEnding|>", "<|Conflict|>","<|Dialogue|>", "<|Foreshadowing|>","<|MoralValue|>", "<|Twist|>"])

def collate_fn(batch):

    # The purpose of the function is the dynamically pad and prepare input batches
    # for training a transformer language model ( gpt-2 in our case)

    # Input:
    #    Batch because we want the sequences of same length within the batch

    # Output:
    #   Dictionary of:
    #        1) input_ids     : Padded input sequences
    #        2) attention_mask: which token consider or not
    #        3) labels        : targets for the language modeling loss (with padding/control tokens ignored)

    #x in batch is a dictionary (keys: input_ids ids_type attention)
    # Pad dynamically to the max length within the *batch*

    input_ids_list = [x['input_ids'] for x in batch]                      # Extracts the list of input_ids from each sample (x) in the batch.

    max_len_sent   = len( max(input_ids_list , key = len))                # Finds the maximum sequence length in this batch.

    input_ids      = torch.tensor(pad_lists(input_ids_list,max_len_sent)) # Uses pad_lists (defined earlier) to pad all sequences to max_len_sent and tranform  it into a tensor


    attention_mask = (input_ids != tokenizer.pad_token_id).long()         # Creation of the attention mask, binary mask, which assign 1 to the real tokens
                                                                          # and 0 to the pad tokens

    # Labels are same as input_ids except for pad tokens
    labels = input_ids.clone()                                            # Initialize the labels equal to the input_ids

    for sp_token in ignore_tokens:
        labels[input_ids == sp_token] = -100                              # Replace all tokens in ignore_tokens in -100 because in pytorch CrossEntropyLoss ignore -100 values

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

# Create a DataLoader for trainin
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)  # Train returned as a dictionary of {input_ids, attention mask, labels}
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)   # Valuation returned as a dictionary of {input_ids, attention mask, labels}

In [ ]:
# Example of a batch in the train laoder

for batch in train_loader:
    #
    print(tokenizer.decode(batch['input_ids'][0])) # Print the first sequence of the batch considered

    print(batch['input_ids'][0])                   # Print of the input_ids of the first sequence of the batch considered

    print(batch['attention_mask'][0])              # Print of the attention mask of the first sequence of the batch considered

    print(batch['labels'][0])                      # Print of the labels of the first sequence of the batch considered

    break

<|Twist|> Once upon a time , a grumpy comet lived in the sky . The comet was grumpy because it was always alone . It wanted to take a friend with it as it flew through the sky . One day , the grumpy comet saw a little bird on Earth . The comet thought the bird could be a good friend . So , it took the bird with it as it flew through the sky . The bird was happy to fly with the comet , and they had fun together . But then , something unexpected happened . The bird started to glow and sparkle like the comet . It turned out that the bird was a magic bird ! The magic bird made the grumpy comet happy , and they became best friends forever . And that is how the grumpy comet found a friend and became a happy comet . <|eos|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <|pad|> <

# Training phase

In [ ]:
def avg_cross_entropy_per_story(outputs, label_ids):

    # Input:
    #    model's outputs obatined after a forward pass into the model
    #    Label_ids, true token ids the model should predict

    # Output:
    #    Average CrossEntropy per story

    logits = outputs.logits                                                              # Raw scores that the model produce before the softmax, the sahpe is [B, S, V] where:
                                                                                         #                                                                                     1) B is the batch size
                                                                                         #                                                                                     2) S is the length of the sequence
                                                                                         #                                                                                     3) V is the length of the vocabulary


    logits = logits.transpose(1, 2)                                                      # Transpose operation for the CrossEntropyLoss, expect a shape of [B, V, S];

    cross_ent_fn = torch.nn.CrossEntropyLoss(reduction='none', ignore_index=-100)        # Define of CrossEntropy, ignoring index -100, reduction = 'none' in order to compute per token loss;

    cross_ent    = cross_ent_fn(logits[:, :, :-1], label_ids[:, 1:])                     # Compute the CrossEntropy loss

    cross_ent    = cross_ent.sum(dim=-1).mean()                                          # Sum across each phrase and then computer the mean;

    return cross_ent.item()                                                              # Convert the TorchTensor into a float;

In [ ]:
def sum_cross_entropy_for_batch(outputs,label_ids):

    # Input:
    #    model's outputs obatined after a forward pass into the model
    #    Label_ids, true token ids the model should predict

    # Output:
    #    Average CrossEntropy per batch

    logits              =  outputs.logits                                                    # Raw scores that the model produce before the softmax, the sahpe is [B, S, V] where:
                                                                                             #                                                                                     1) B is the batch size
                                                                                             #                                                                                     2) S is the length of the sequence
                                                                                             #                                                                                     3) V is the length of the vocabulary

    logits              =  logits.transpose(1, 2)                                            # Transpose operation for the CrossEntropyLoss, expect a shape of [B, V, S];

    cross_ent_fn        =  torch.nn.CrossEntropyLoss(reduction='none', ignore_index=-100)    # Define of CrossEntropy, ignoring index -100, reduction = 'none' in order to compute per token loss;

    cross_ent           =  cross_ent_fn(logits[:, :, :-1], label_ids[:, 1:])                 # Compute the CrossEntropy loss

    cross_ent_per_token =  cross_ent.sum()                                                   # Compute the loss per batch

    return cross_ent_per_token.item()                                                        # Convert the TorchTensor into a float;


In [ ]:
def train(model, train_loader, val_loader, optimizer, batch_size, epochs, device):

    # Input:
    #    model        : gpt-2 model thta will be trained
    #    train_loader : Obtained from DataLoader, provides batches with inside the disctionaries "input_ids", "attention_mask", "labels"
    #    val_loader   : Obtained from Dataloader, used for initial evaluation
    #    optimizer    : Optimizer used for the tyrainin phase
    #    batch_size   : size of the Batch
    #    epochs       : Number of epochs for the training phase
    #    device       : GPU, if available, or CPU

    # Output:
    #     Print of the loss and cross entropy

    model.to(device)
    print("Initial Evaluation:")
    evaluate(model, val_loader, device)

    cross_ent_fn = torch.nn.CrossEntropyLoss(reduction='none', ignore_index=-100)                    # Setting of the CrossEntropy
    log_frequency = 100
    for epoch in range(epochs):
       #
        model.train()

        print(f"==== Epoch {epoch + 1} ====")

        train_loss        = 0.0
        train_cross_ent   = 0.0
        running_loss      = 0.0
        running_cross_ent = 0.0

        for idx, batch in enumerate(train_loader):
            #
            input_ids        = batch["input_ids"].to(device)                                          # Extract the input_ids from batch and move it to the device

            attention_mask   = batch["attention_mask"].to(device)                                     # Extract the input_ids from batch and move it to the device

            label_ids        = batch["labels"].to(device)                                             # Extract the input_ids from batch and move it to the device

            optimizer.zero_grad()                                                                     # Clear the previous gradient to avoid accomulations

            outputs          = model(input_ids, attention_mask=attention_mask, labels=label_ids )     # Compute the outputs throught a forward run into the model, outputs contains
                                                                                                      # the loss

            loss             = outputs.loss                                                           # Average loss per token in batch

            train_loss       += loss.item()                                                           # Accumulate the losses for each batch

            loss.backward()                                                                           # Compute gradients via backpropagation
            optimizer.step()                                                                          # Update the model weights

            cross_ent         = avg_cross_entropy_per_story(outputs, label_ids)                       # Compute our average cross-entropy per story (higher-level metric).
            train_cross_ent   += cross_ent

            running_loss      += loss.item()
            running_cross_ent += cross_ent

            if (idx+1) % log_frequency == 0:
                #
                print(f"Batch {idx+1}: loss = {running_loss / log_frequency} --- cross_ent = {running_cross_ent / log_frequency}")

                running_loss      = 0.0                                                               # Re-initialize to zero after the print

                running_cross_ent = 0.0                                                               # Re-initialize to zero after the print

    # It will be printed an estimate of the loss mean per token and for story ( mean of the mean)
    # It could be the exact mean if every batch would be made by the same number of stories
    print(f"TRAINING after Epoch {epoch + 1}: avg loss per token = {train_loss / len(train_loader):.4f} --- avg cross entropy per story = {train_cross_ent / len(train_loader):.4f}")

    print(f"Evaluation after Epoch: {epoch + 1} ")

    evaluate(model, val_loader, device)                                                               # Evaluation after the epochs



def evaluate(model, loader, device):
    #
    model.to(device)

    model.eval()                                                                         # Set the model to evaluation mode in order to disables droputs and batch normalization

    val_loss = 0.0                                                                       # Iniatialize the total loss to zero

    for batch in loader:
        #
        input_ids = batch["input_ids"].to(device)                                        # Extract the input_ids from batch and move it to the device

        attention_mask = batch["attention_mask"].to(device)                              # Extract the attention mask from batch and move it to the device

        label_ids = batch["labels"].to(device)                                           # Extract the label_ids from batch and move it to the device

        with torch.no_grad():
            #
            outputs = model(input_ids, attention_mask=attention_mask, labels=label_ids)  # Compute the outputs of the model

            generic_loss = sum_cross_entropy_for_batch(outputs,label_ids)                # Batch loss computed from function above

            val_loss += generic_loss                                                     # Accumulation of the loss

    num_non_pad_tokens = sum(len(t['labels'][t['labels'] != -100]) for t in loader)      # Total number of non - pad tokens


    print(f"VALIDATION: loss = {val_loss / num_non_pad_tokens:.4f} --- avg cross entropy = {val_loss / len(val_dataset):.4f} --- Perplexity per token = {math.exp(val_loss / num_non_pad_tokens):.4f}")

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)                                                                                           # Set of Adam Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

file_id     = "1BiBGLdcFtRs8x71ym9V1q2LNpl-cWxzN"
url         = f"https://drive.google.com/uc?id={file_id}"

output_name = "modello_piccolo_voc8192_ep2_256_2_4.pth"

gdown.download(url, output = output_name, quiet = False)

if do_train:
    train(model, train_loader, val_loader, optimizer, batch_size, epochs, device)                                                                    # If do_train --> True , we do the training phase
else:
  if device == torch.device("cuda"):

    model.load_state_dict( torch.load(output_name,weights_only=True) )

  else:

   model.load_state_dict( torch.load(output_name, cmap_location=torch.device('cpu'), weights_only=True) )                                                                  # If do_train --> False we load the training results from Google drive
#

Downloading...
From: https://drive.google.com/uc?id=1BiBGLdcFtRs8x71ym9V1q2LNpl-cWxzN
To: /content/modello_piccolo_voc8192_ep2_256_2_4.pth
100%|██████████| 16.8M/16.8M [00:00<00:00, 30.2MB/s]


# Test phase

In [ ]:
test_dataset = test_dataset.map(preprocess)

test_loader=DataLoader(test_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

evaluate(model,test_loader,device)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


VALIDATION: loss = 1.8248 --- avg cross entropy = 352.7103 --- Perplexity per token = 6.2015


# Generation of the story

In [ ]:
def manual_generate(model,device,input,max_lenght,greedy=False):

    # Input:
    #   model     : The trained gpt-2 model for generating text;
    #   device    : GPU if available or CPU;
    #   input     : The initial string or text from which we generate;
    #   max_length: Maximum number of tokens to generate
    #   Greedy    : Boolean variable, if True --> token with highest value to be chosen, else choose from distribution

    # Output:
    #    generates a text sequence from a given prompt using gpt-2 model
    #    It's a manual implementation of greedy decoding or sampling, without relying on HuggingFace’s generate().


    model.to(device)
    model.eval()

    input_ids=tokenizer(input)['input_ids']                                             # Tokenization of the input

    while len(input_ids) < max_length:
        #
        x = torch.tensor(input_ids).unsqueeze(0).to(device)                             # Adding one dimension to input_ids

        with torch.no_grad():
          #
          output = model(x)
          logits = output.logits[0,-1,:]                                                # Select the last logits to predict next token, logits has shape [1, sequence_length, vocab_size].

          if greedy:
              #
              next_id = logits.argmax().item()                                          # Token prediction selecting the argmax()
          else:
              #
              next_id = torch.multinomial(torch.softmax(logits, dim = 0), 1).item()     # Token prediction using softmax

          if next_id == tokenizer.eos_token_id or next_id == tokenizer.pad_token_id:    # Break if special token
              break

          input_ids.append(next_id)

    return tokenizer.decode(input_ids)

In [ ]:
features = ["BadEnding", "Conflict", "Dialogue", "Foreshadowing", "MoralValue", "Twist"]

# Function to generate text
def generate_story(model, device, story_features, input_text="", max_length=400, auto=True):

    # Input:
    #    model           : The trained gpt-2 model for generating text;
    #    device          : GPU if available or CPU;
    #    story_feature   : One of the tag-features above;
    #    input_text      : Optional starting text to generate the story;
    #    max_length      : Maximum number of tokens to generate;
    #    Auto            : Boolean variable, if True --> auto generation, else manual generation function above;

    # Output:
    #    Generation of the story



    # Construct the initial prompt including feature tokens
    # Follow the order that the model is used to
    prefix = ""
    for feat in features:
        if feat in story_features:
            prefix += "<|"+feat+"|>" + " "                                                     # Append of the special token at the beginning;

    input_text = prefix + input_text                                                           # Special token added with the input_text
    print(f"Input text: {input_text}")

    if auto:
        #
        model.to(device)
        model.eval()

        input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device)               # Encode of the input_text and move to the device

        # Generate text
        with torch.no_grad():
            output_sequences = model.generate(

                input_ids=input_ids,

                max_length=max_length,

                do_sample=True,

                top_p=0.9,                                                                      # Top-p (nucleus) sampling to filter top 90% of cumulative probability

                temperature=0.6                                                                 # Temperature parameter, for less probable tokens
                )

        generated_story = tokenizer.decode(output_sequences[0], skip_special_tokens=False)      # Decode of the generated sequence using the existing one
    else:
        generated_story = manual_generate(model, device, input_text, max_length)


    for feat in features:
        if generated_story.startswith("<|"+feat+"|>"):
            generated_story = generated_story[len(feat)+4:]                                     # Remove special token + <| |> from the beginning of the generated story
            generated_story = generated_story.lstrip()                                          # remove initial whitespaces

    return generated_story

# Analysis on the story generated

In [ ]:
# Generation of a story starting from a no-sense question in order to see if we obtain an answer with sense or not

generate_story(model, device, ["Dialogue"],"Tom asked his mom: \"Do sharks fly?\" ",400, True)

Input text: <|Dialogue|> Tom asked his mom: "Do sharks fly?" 


'Tom asked his mom : " Do sharks fly ?" His mom said , " Yes , I believe you can do it . Let \' s go find a big surprise !" Tom was so excited . He ran to the store and saw a big jar of jelly . He wanted to eat it . He reached out and grabbed the jar . He put it in his pocket and ran to the store . He grabbed the jar and took it out . He was so happy . He was so happy . He ate the jelly and shared it with his mom . He was so proud of himself . <|eos|>'

In [ ]:
# Generation of a story starting from a question on why we need to do something, we want to see if we obtain good results or not

generate_story(model, device, ["Dialogue"],"Tom asked his mom: \"Why do I have to go to school?\" ",400, False)

Input text: <|Dialogue|> Tom asked his mom: "Why do I have to go to school?" 


'Tom asked his mom : " Why do I have to go to school ?" His mom replied : " Or bees on the stage !" Tom smiled and said : " I want to go and school ." His mom smiled and said : " Okay , it was too hot . Can we go to school ? I want nice with you at school . Can you please ?" Tom nodded and took Mom on his shoes . He climbed up the bench and rode to school . He felt the soft sand and cozy . He felt proud he was going to school with . Mom was slow and not scared . She said : " Okay , Tom , you can . But be careful . The stairs is fun and tall . It \' s heavy ." Lily climbed and reached up high . He sat on the chair and reads . She dreamed of the stairs with her mom and dad . She ran to Tom and gave him a hug . Tom felt cold and cozy . They had fun together . They didn \' t mind . The two of them shared to the kids . They were happy and healthy .'

In [ ]:
# Analysis about creativity

def token_unicity_rate(story):
    tokens=tokenizer.encode(story)
    unique_tokens=set(tokens)
    return (float(len(unique_tokens))/float(len(tokens)))

number_stories=10
unicity_rate=0.0
for i in range(number_stories):
    i = random.randint(0, 5)
    unicity_rate+=token_unicity_rate(generate_story(model, device, [features[i]],"",100, True)) #we use short stories
print(f"Unicity_rate = {unicity_rate/number_stories:.4f}")

Input text: <|Conflict|> 
Input text: <|Conflict|> 
Input text: <|Foreshadowing|> 
Input text: <|Foreshadowing|> 
Input text: <|MoralValue|> 
Input text: <|MoralValue|> 
Input text: <|Foreshadowing|> 
Input text: <|BadEnding|> 
Input text: <|Twist|> 
Input text: <|BadEnding|> 
Unicity_rate = 0.5141


In [ ]:
# Generation of a story starting from one of the six feature tag

generate_story(model, device, ["Dialogue"],"",400, True)

Input text: <|Dialogue|> 


'Once upon a time , there was a little girl named Lucy . She loved to play outside in the sun . One day , she saw a big tree with a hole . Lucy wanted to see what was inside . She ran to the tree and saw a little girl named Lily . Lily was sad because she lost her toy . Lily wanted to help Lily find her toy . She said , " Let \' s look for the toy together !" Lily and Lily looked everywhere . They looked under the tree , behind the tree , and even in the hole . Finally , they found the toy under a bush . Lily was so happy ! She said , " Thank you , Lily !" They played together all day and had lots of fun . <|eos|>'